# Inspect and prepare the 936-pair dataset

Select a GPU runtime on Colab. Copy the repository source and `data/936` to Drive, set `PROJECT_DIR` below, and run all cells. Locally select `.venv`. Run this notebook before the training notebook. Masks must be indexed IDs, not normalized.

**Label audit:** The local masks store only IDs 0/1/2. A closely matching [Tobacco Aerial Dataset](https://data.mendeley.com/datasets/5dpc5gbgpz/2) documents 0=background, 1=crop, 2=weed, including a 936-image campaign at 480x352 with the same folder layout. This is a provisional match, not proof of the local archive identity. Its crop is tobacco, so do not claim wheat identity without dataset confirmation. Override CLASS_NAMES if your instructor supplies the mapping.


In [ ]:
import os, sys, subprocess
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Copy main.py, data.py, model.py and the dataset into this Drive folder.
    PROJECT_DIR = Path('/content/drive/MyDrive/unet')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'scikit-learn>=1.4,<2', 'Pillow', 'matplotlib'])
    if not (PROJECT_DIR / 'main.py').exists():
        raise FileNotFoundError('Set PROJECT_DIR to the Drive folder containing main.py, data.py, model.py')
    os.chdir(PROJECT_DIR)
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))
else:
    PROJECT_DIR = Path.cwd()
print('Project:', PROJECT_DIR)
print('Python:', sys.executable)


In [ ]:
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from data import prepare_dataset, make_dataset
from main import configure_gpu
configure_gpu(require_gpu=False)
DATASET = Path('data/936')
SEED = 42
LIMIT = None
images, masks, manifest = prepare_dataset(DATASET, LIMIT, seed=SEED)
if len(images) != 936:
    raise ValueError(f'The assignment expects 936 pairs; found {len(images)}')
print('Development pool:', len(manifest['train']) + len(manifest['validation']))
print('Split sizes:', {k:len(v) for k,v in manifest.items()})
print('IDs 1 and 2 have no class-name metadata: do not guess their semantic labels.')


In [ ]:
ids = manifest['train'][:5]
fig, axes = plt.subplots(len(ids), 2, figsize=(9, 3 * len(ids)), squeeze=False)
for row, key in enumerate(ids):
    with Image.open(images[key]) as im:
        axes[row,0].imshow(im.convert('RGB'))
    with Image.open(masks[key]) as mask:
        array = np.asarray(mask)
        print(key, 'IDs:', np.unique(array).tolist())
        axes[row,1].imshow(array, cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
    for col,title in enumerate((f'Image {key}', 'Class-ID mask')):
        axes[row,col].set_title(title); axes[row,col].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
x,y = next(iter(make_dataset(images,masks,manifest['train'],size=128,batch_size=2,one_hot=True)))
print('RGB input:', x.shape, x.dtype)
print('One-hot target:', y.shape, y.dtype)
assert np.allclose(y.numpy().sum(-1),1)
output=Path('runs'); output.mkdir(exist_ok=True)
(output/'prepared_dataset.json').write_text(json.dumps({
    'dataset':str(DATASET.resolve()),'limit':LIMIT,'seed':SEED,'manifest':manifest},indent=2))
fig.savefig(output/'prepared_dataset_preview.png',dpi=140)
print('Preparation complete. Next run trainUnet.ipynb.')
